# Laboratorio 9 — Embeddings

**Natural Language Processing**

En este laboratorio se construyen y comparan embeddings de palabras en español:

- **Corpus principal:** Spanish News Classification (`df_total.csv`), para entrenar embeddings y clasificar noticias.
- **Segundo corpus:** *Don Quijote de la Mancha* (`don-quijote.txt`), para comparar cómo cambia el significado según el texto.
- **Vectores preentrenados:** fastText SUC (tamaño M, 100 dimensiones).

In [1]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import gensim
import sklearn
import torch

# Semilla para que los resultados sean reproducibles
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Versiones (se usan para el requirements.txt)
print("gensim", gensim.__version__)
print("scikit-learn", sklearn.__version__)
print("torch", torch.__version__)

gensim 4.4.0
scikit-learn 1.8.0
torch 2.14.1


# 1. Preparación de los corpus

### 1.1 Carga de los datos

Cargamos las noticias y el Quijote. Del Quijote quitamos el texto de Project Gutenberg y dejamos solo la novela.

In [ ]:
# Corpus de noticias
df_news = pd.read_csv("df_total.csv")
print("Noticias:", df_news.shape)
print(df_news["Type"].value_counts())

# Corpus del Quijote (utf-8-sig quita el BOM del inicio del archivo)
with open("don-quijote.txt", encoding="utf-8-sig") as f:
    raw_quijote = f.read()

# Nos quedamos solo con la novela: desde el título hasta el aviso final de Gutenberg
inicio = raw_quijote.index("El ingenioso hidalgo don Quijote de la Mancha")
fin = raw_quijote.index("End of Project Gutenberg's Don Quijote")
quijote = raw_quijote[inicio:fin].strip()

print(f"\nQuijote: {len(quijote):,} caracteres")
print(quijote[:300])
print("...")
print(quijote[-200:])

### 1.2 Oraciones y tokens

Separamos cada corpus en oraciones y cada oración en palabras, en orden, porque gensim espera una lista de listas de tokens.

En las noticias falta el espacio después de muchos puntos (`agregó.El`), así que lo agregamos antes de separar las oraciones.

In [5]:
import re
import nltk
from nltk.tokenize import sent_tokenize

nltk.download("punkt_tab", quiet=True)

TOKEN_RE = re.compile(r"\w+")


def segmentar(texto):
    """Divide un texto en oraciones."""
    # Agrega el espacio que falta después de . ! ? cuando sigue una mayúscula (p. ej. "agregó.El")
    texto = re.sub(r"(?<=\w)([.!?])(?=[A-ZÁÉÍÓÚÑ¿¡])", r"\1 ", texto)
    # Une los saltos de línea para no partir oraciones
    texto = re.sub(r"\s+", " ", texto)
    return sent_tokenize(texto, language="spanish")


def tokenizar(oracion):
    """Devuelve los tokens de una oración en orden (sin puntuación)."""
    return TOKEN_RE.findall(oracion)


# Noticias: cada noticia se segmenta por separado
oraciones_news = []
for texto in df_news["news"]:
    for oracion in segmentar(texto):
        tokens = tokenizar(oracion)
        if tokens:  # descarta "oraciones" que eran solo puntuación
            oraciones_news.append(tokens)

# Quijote
oraciones_quijote = [tokenizar(o) for o in segmentar(quijote)]
oraciones_quijote = [t for t in oraciones_quijote if t]

print(f"Noticias: {len(oraciones_news):,} oraciones")
print(f"Quijote:  {len(oraciones_quijote):,} oraciones")
print("\nEjemplo noticias:", oraciones_news[0])
print("Ejemplo Quijote: ", oraciones_quijote[100])

Noticias: 21,805 oraciones
Quijote:  9,513 oraciones

Ejemplo noticias: ['Durante', 'el', 'foro', 'La', 'banca', 'articulador', 'empresarial', 'para', 'el', 'desarrollo', 'sostenible', 'el', 'director', 'de', 'sostenibilidad', 'y', 'clientes', 'globales', 'de', 'BBVA', 'en', 'Colombia', 'Andrés', 'García', 'aseguró', 'que', 'es', 'importante', 'entender', 'que', 'la', 'sostenibilidad', 'no', 'la', 'podemos', 'asociar', 'a', 'mayores', 'costos']
Ejemplo Quijote:  ['No', 'puedo', 'ser', 'tu', 'igual', 'que', 'este', 'decoro', 'se', 'debe', 'a', 'tus', 'proezas', 'y', 'a', 'tu', 'fama', 'puesto', 'que', 'como', 'yo', 'perdiste', 'el', 'seso']


### 1.3 Vocabulario de fastText SUC

Revisamos si SUC distingue mayúsculas y tildes, para normalizar nuestros corpus de la misma forma.

In [6]:
from gensim.models import KeyedVectors

# El archivo pesa 1.1 GB: se guarda fuera del repositorio y no se sube a GitHub
SUC_PATH = "/Users/michelle/Downloads/embeddings-m-model.vec"
suc = KeyedVectors.load_word2vec_format(SUC_PATH, limit=200_000)
print("Vocabulario cargado:", len(suc.key_to_index))

for palabra in ["España", "españa", "espana", "Gobierno", "gobierno", "economía", "economia", "2020", "BBVA", "bbva"]:
    print(f"{palabra:<10}", palabra in suc.key_to_index)

Vocabulario cargado: 200000
España     False
españa     True
espana     True
Gobierno   False
gobierno   True
economía   True
economia   True
2020       True
BBVA       False
bbva       True


### 1.4 Normalización

- **Minúsculas:** igual que SUC, para que "Gobierno" y "gobierno" sean la misma palabra.
- **Tildes:** se conservan, porque cambian el significado ("sé" / "se", "más" / "mas").
- **Puntuación:** ya se quitó al tokenizar.
- **Números:** se quitan los tokens que son solo dígitos; aportan poco al significado de las palabras.
- **Stopwords y lematización:** no se aplican. Quitar stopwords cambia qué palabras quedan juntas en la ventana de contexto, y lematizar junta "gobiernos" y "gobernó" en una sola forma, perdiendo información de número y tiempo verbal.

In [7]:
def normalizar(tokens):
    """Pasa a minúsculas y quita los tokens que son solo números."""
    return [t.lower() for t in tokens if not t.isdigit()]


corpus_news = [normalizar(o) for o in oraciones_news]
corpus_news = [o for o in corpus_news if o]

corpus_quijote = [normalizar(o) for o in oraciones_quijote]
corpus_quijote = [o for o in corpus_quijote if o]

print("Antes:  ", oraciones_news[0][:12])
print("Después:", corpus_news[0][:12])

Antes:   ['Durante', 'el', 'foro', 'La', 'banca', 'articulador', 'empresarial', 'para', 'el', 'desarrollo', 'sostenible', 'el']
Después: ['durante', 'el', 'foro', 'la', 'banca', 'articulador', 'empresarial', 'para', 'el', 'desarrollo', 'sostenible', 'el']
